In [3]:
import csv
import re
import numpy as np
from cds2py import plot_svg
from datetime import datetime, date
from collections import namedtuple
from slugify import slugify
from quantiphy import Quantity
from tabulate import tabulate

In [4]:
with open('diet.csv') as csvfile:
    data = list(csv.reader(csvfile, delimiter=','))

DietEntry = namedtuple("DietEntry", [slugify(elem).replace("-","_") for elem in data[0]])

def quantify_elem(elem:str, field:str):
    if field == "date":
        return datetime.strptime(elem,"%Y-%m-%d")
    elif field == "completed":
        return elem == "true"
    elif field.split("_")[-1] == "mg":
        return Quantity(float(elem)*1e-3, units="g")
    elif field.split("_")[-1] == "ug":
        return Quantity(float(elem)*1e-6, units="g")
    elif field.split("_")[-1] == "iu":
        return Quantity(float(elem)*1e-6/40, units="g")
    elif field.split("_")[-1] == "kcal":
        return Quantity(float(elem)*1e3, units="cal")
    else:
        return Quantity(elem, units=field.split("_")[-1])

quantities = [
    DietEntry(
        *[
            quantify_elem(value, DietEntry._fields[index])
            for index,value in enumerate(elem)
        ]
    ) for elem in data[1:]
]


In [ ]:
darray = np.array(quantities)[:,1:-1].astype(float)
Names = [re.sub(" [kmiu]"," ", elem.replace("_"," ")).replace(" u"," g") for elem in DietEntry._fields[1:-1]]
metric_table = np.vstack(
        ( Names, [ Quantity(elem).render(prec=2) for elem in darray.mean(axis=0) ], [ Quantity(elem).render(prec=2) for elem in darray.std(axis=0) ]  )
    )

with open("diet.table","w") as mdfile:
    mdfile.write(
        tabulate( metric_table.T, headers=["Metric", "Average", "Deviation"],)
    )

In [34]:
import importlib.resources as rsc
import json
import re
import nutrimetrics.resources.dri

src_txt = rsc.read_text(nutrimetrics.resources.dri,"ear-male.json")
average_requirement = json.loads(re.sub("//.*","",src_txt))["dietary_reference_intakes"]
src_txt = rsc.read_text(nutrimetrics.resources.dri,"rda-male.json")
average_allowance = json.loads(re.sub("//.*","",src_txt))["dietary_reference_intakes"]


In [37]:

for name, average, _ in metric_table.T:
    nutrient = name.replace(" ","-").replace("-g","")
    if nutrient in average_requirement:
        print(nutrient)
        # print(Quantity(average_requirement[nutrient],units="g")/Quantity(average, units="g"))

folate
vitamin-a
vitamin-c
vitamin-d
vitamin-e
calcium
copper
iron
magnesium
manganese
phosphorus
potassium
selenium
sodium
zinc
